In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import seaborn as sns
from scipy.special import softmax
from scipy.stats import gumbel_r
from scipy.stats import norm
from scipy.special import logit

import utilities as utils

In [ ]:
### operational parameters ###

savedir = "./Figures/"
savefigflag = False
savedataflag = True # save data produced by this notebook
loaddataflag = True # load data already produced by this notebook, instead of re-running analyses

beads_datadir = "./Data/beads/"
horses_datadir = "./Data/horses/"

# number of threads for parallelized analyses
N_threads = 22


### parameters governing analyses ###

# number of beads to include in X, from current trial backward (bead-prediction experiment) for "fully-optimal" strategy
wsize = 7

# limit for the number of iterations for the IB algorithm
IB_iterlimit = 100000
N_b_ib = 1000 # number of beta points to sample for the standard IB algorithm
max_b_ib = 50 # maximum beta value to sample for the standard IB algorithm
N_b_softmax = 10000 # number of betastar points to sample for the softmax solution
max_b_softmax = 100 # maximum betastar value to sample for the softmax solution


### fixed parameters for the generative structure of the experiments ###

# generative parameters for the bead-prediction experiment
h_=0.99 # jar stay probability
p0_=0.8 # probability of drawing bead type 0 from jar 0
p1_=0.2 # probability of drawing bead type 0 from jar 1
P0 = np.array([[0.5, 0.5]]) # prior over jars for computing posterior probabilities over hidden markov process
H = np.ones((2,2)) - np.abs(np.eye(2)*-1 + h_) # transition matrix
E = np.vstack((np.array([[1,0]])*p0_ + np.array([[0,1]])*(1-p0_),np.array([[1,0]])*p1_ + np.array([[0,1]])*(1-p1_))) # emission matrix

# generative parameters for the horse prediction experiments
paramdict = {
    'lowWS': {
        'weakLLR': 0.45,
        'WSratio': 1.3,
        'p1': 0.06
    },
    'midWS': {
        'weakLLR': 0.2,
        'WSratio': 2.5,
        'p1': 0.08
    },
    'highWS': {
        'weakLLR': 0.18,
        'WSratio': 6.3,
        'p1': 0.02
    }
}

# p(Y) for both tasks
p_Y = np.array([[0.5, 0.5]])


### helper functions ###

def savefig(fig, name, ftype="svg", savefigflag=savefigflag, savedir=savedir):
    if savefigflag:
        fig.savefig(savedir + name + "." + ftype, bbox_inches="tight", dpi=300)

def format_axis(ax):
    ax.tick_params(axis='x', labelsize=5, width=0.5, length=2)
    ax.tick_params(axis='y', labelsize=5, width=0.5, length=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['left'].set_linewidth(0.5)

In [ ]:
def get_beta_matched_bounds(p_XgY,p_Y,N_b=1000,max_b=100,betastar_array=None,iterlimit=100000,N_inits=3,N_threads=1,base_seed=209,enforce_monotonic=False):
    softmax_curve = utils.get_softmax_IB_curve(p_XgY,p_Y,N_b=N_b,max_b=max_b,betastar_array=betastar_array)
    beta_array = softmax_curve['beta']
    betastar_array = softmax_curve['betastar']
    IB_bound_randinit = utils.get_IB_bound(p_XgY,p_Y,N_b=N_b,max_b=max_b,iterlimit=iterlimit,beta_array=beta_array,init='random',N_inits=N_inits,betastar_array=None,N_threads=N_threads,base_seed=base_seed,enforce_monotonic=enforce_monotonic)
    IB_bound_softmaxinit = utils.get_IB_bound(p_XgY,p_Y,N_b=N_b,max_b=max_b,iterlimit=iterlimit,beta_array=beta_array,init='softmax',N_inits=N_inits,betastar_array=betastar_array,N_threads=N_threads,base_seed=base_seed,enforce_monotonic=enforce_monotonic)
    return softmax_curve, IB_bound_randinit, IB_bound_softmaxinit



In [ ]:
def evaluate_IB_bound_v3(X,P_XgY,P_RgX=None,p_Y=None,N_b=1000,max_b=50,iterlimit=10000000,N_inits=3,N_threads=1):

    # def compute_IB_fn(beta,p_RgX,p_X,p_R,p_YgR,p_Y):
    #     Ixr = np.sum(p_RgX*p_X*np.log2(p_RgX/p_R))
    #     Iry = np.sum(p_YgR*p_R*np.log2(p_YgR/p_Y.reshape(-1,1)))
    #     return Ixr - beta*Iry

    base_seed = 209

    p_XgY = P_XgY(X)
    N_y = p_XgY.shape[1]
    N_x = p_XgY.shape[0]

    if p_Y is None:
        p_Y = np.array([1/N_y]*N_y)

    p_XY = p_XgY * p_Y
    p_X = np.sum(p_XY, axis=1, keepdims=True)
    p_YgX = p_XY / p_X
    
    if P_RgX is None:
        P_RgX = lambda betastar, X, **probs : np.exp(betastar*probs['p_YgX']) / np.sum(np.exp(betastar*probs['p_YgX']),axis=1,keepdims=True)

    # betastar_array = np.linspace(0.1,max_b,N_b)
    betastar_array = np.linspace(max_b/N_b,max_b,N_b)

    softmax_dict = {
        'I_XR': np.zeros_like(betastar_array),
        'I_RY': np.zeros_like(betastar_array),
        'betastar': betastar_array,
        'beta': np.zeros_like(betastar_array),
        'accuracy': np.zeros_like(betastar_array),
        'alpha': np.zeros_like(betastar_array),
        'H_R': np.zeros_like(betastar_array),
        'p_YeR': np.zeros((len(betastar_array), 1)),
        'p_YneR': np.zeros((len(betastar_array), 1))
    }

    IB_dict_randinit = {
        'I_XR': np.zeros_like(betastar_array),
        'I_RY': np.zeros_like(betastar_array),
        'beta':  np.zeros_like(betastar_array),
        'pseudo_accuracy': np.zeros_like(betastar_array),
        'alpha': np.zeros_like(betastar_array),
        'betastar': np.zeros_like(betastar_array),
        'H_R': np.zeros_like(betastar_array),
        'p_YeR_mapped': np.zeros((len(betastar_array),N_y)),
        'p_YneR_mapped': np.zeros((len(betastar_array),(N_y**2-N_y))),
        'p_RgX_maxdiff': np.zeros_like(betastar_array)
    }

    IB_dict_softmaxinit = {
        'I_XR': np.zeros_like(betastar_array),
        'I_RY': np.zeros_like(betastar_array),
        'beta':  np.zeros_like(betastar_array),
        'pseudo_accuracy': np.zeros_like(betastar_array),
        'alpha': np.zeros_like(betastar_array),
        'betastar': np.zeros_like(betastar_array),
        'H_R': np.zeros_like(betastar_array),
        'p_YeR_mapped': np.zeros((len(betastar_array),N_y)),
        'p_YneR_mapped': np.zeros((len(betastar_array),(N_y**2-N_y))),
        'p_RgX_maxdiff': np.zeros_like(betastar_array)
    }

    p_RgX_softmax = np.zeros((N_b,N_x,N_y))

    for ib, betastar in enumerate(betastar_array):

        # for this value of betastar, compute p(R|X) and find I(X;R) and I(R;Y)
        # using the softmax formulation
        p_RgX1 = P_RgX(betastar,X,p_YgX=p_YgX)
        p_RX1 = p_RgX1*p_X
        p_R1 = np.sum(p_RX1, axis=0, keepdims=True)
        p_RgY1 = p_RgX1.T @ p_XgY

        accuracy1 = np.trace(p_RgY1) / N_y # assumes p(Y) is uniform
        alpha1 = np.log(accuracy1/((1-accuracy1)/(N_y-1)))
        beta = betastar / alpha1

        p_RgX_softmax[ib,:,:] = p_RgX1
        softmax_dict['accuracy'][ib] = accuracy1
        softmax_dict['beta'][ib] = beta
        softmax_dict['alpha'][ib] = alpha1
        softmax_dict['H_R'][ib] = -np.sum(p_R1 * np.log2(p_R1))
        softmax_dict['p_YeR'][ib] = accuracy1
        softmax_dict['p_YneR'][ib] = (1-accuracy1)/(N_y-1)
        softmax_dict['I_XR'][ib] = np.sum(p_RgX1*p_X*np.log2(p_RgX1/p_R1))
        softmax_dict['I_RY'][ib] = np.sum(p_RgY1*p_Y.reshape(1,-1)*np.log2(p_RgY1/p_R1.reshape(-1,1)))

    beta_array = softmax_dict['beta']
    # beta, p_XgY, p_Y, iterlimit, init, N_inits, betastar, base_seed, ib_num = args
    # return [I_XR, I_RY, beta, pseudo_accuracy, alpha, betastar, H_R, p_YeR_mapped, p_YneR_mapped, p_RgX]
    # return [I_XR, I_RY, beta, pseudo_accuracy, alpha, betastar, H_R, p_YeR_mapped, p_YneR_mapped, p_RgX]

    # using the value of beta extracted from the above betastar and accuracy,
    # solve the IB optimization problem using the original numerical algorithm
    # random initialization
    init = 'random'
    with mp.Pool(processes=N_threads) as pool:
        results = [pool.apply_async(solve_IB_for_beta, args=((beta_array[ib], p_XgY, p_Y, iterlimit, init, N_inits, betastar_array[ib], base_seed, ib),)) for ib in range(N_b)]
        results = [res.get() for res in results]
    I_XR = [res[0] for res in results]
    I_RY = [res[1] for res in results]
    betas = [res[2] for res in results]
    pseudo_accuracies = [res[3] for res in results]
    alphas = [res[4] for res in results]
    betastars = [res[5] for res in results]
    H_Rs = [res[6] for res in results]
    p_YeR_mappeds = [res[7] for res in results]
    p_YneR_mappeds = [res[8] for res in results]
    p_RgXs = [res[9] for res in results]

    sort_indices = np.argsort(betastar_array)
    IB_dict_randinit['I_XR'] = np.array(I_XR)[sort_indices]
    IB_dict_randinit['I_RY'] = np.array(I_RY)[sort_indices]
    IB_dict_randinit['beta'] = np.array(betas)[sort_indices]
    IB_dict_randinit['pseudo_accuracy'] = np.array(pseudo_accuracies)[sort_indices]
    IB_dict_randinit['alpha'] = np.array(alphas)[sort_indices]
    IB_dict_randinit['betastar'] = np.array(betastars)[sort_indices]
    IB_dict_randinit['H_R'] = np.array(H_Rs)[sort_indices]
    IB_dict_randinit['p_YeR_mapped'] = np.array(p_YeR_mappeds)[sort_indices,:]
    IB_dict_randinit['p_YneR_mapped'] = np.array(p_YneR_mappeds)[sort_indices,:]
    p_RgXs_sorted = np.array(p_RgXs)[sort_indices]
    p_RgXs_maxinds = [np.argmax(np.abs(p_RgX_softmax[ib] - p_RgXs_sorted[ib])) for ib in range(N_b)]
    IB_dict_randinit['p_RgX_maxdiff'] = np.array([p_RgX_softmax[ib].flatten()[p_RgXs_maxinds[ib]] - p_RgXs_sorted[ib].flatten()[p_RgXs_maxinds[ib]] for ib in range(N_b)])
    IB_dict_randinit['IB_obj_fn'] = IB_dict_randinit['I_XR'] - IB_dict_randinit['beta'] * IB_dict_randinit['I_RY']

    # using the value of beta extracted from the above betastar and accuracy,
    # solve the IB optimization problem using the original numerical algorithm
    # initialize with softmax solution
    init = 'softmax'
    with mp.Pool(processes=N_threads) as pool:
        results = [pool.apply_async(solve_IB_for_beta, args=((beta_array[ib], p_XgY, p_Y, iterlimit, init, N_inits, betastar_array[ib], base_seed, ib),)) for ib in range(N_b)]
        results = [res.get() for res in results]
    I_XR = [res[0] for res in results]
    I_RY = [res[1] for res in results]
    betas = [res[2] for res in results]
    pseudo_accuracies = [res[3] for res in results]
    alphas = [res[4] for res in results]
    betastars = [res[5] for res in results]
    H_Rs = [res[6] for res in results]
    p_YeR_mappeds = [res[7] for res in results]
    p_YneR_mappeds = [res[8] for res in results]
    p_RgXs = [res[9] for res in results]

    sort_indices = np.argsort(betastar_array)
    IB_dict_softmaxinit['I_XR'] = np.array(I_XR)[sort_indices]
    IB_dict_softmaxinit['I_RY'] = np.array(I_RY)[sort_indices]
    IB_dict_softmaxinit['beta'] = np.array(betas)[sort_indices]
    IB_dict_softmaxinit['pseudo_accuracy'] = np.array(pseudo_accuracies)[sort_indices]
    IB_dict_softmaxinit['alpha'] = np.array(alphas)[sort_indices]
    IB_dict_softmaxinit['betastar'] = np.array(betastars)[sort_indices]
    IB_dict_softmaxinit['H_R'] = np.array(H_Rs)[sort_indices]
    IB_dict_softmaxinit['p_YeR_mapped'] = np.array(p_YeR_mappeds)[sort_indices,:]
    IB_dict_softmaxinit['p_YneR_mapped'] = np.array(p_YneR_mappeds)[sort_indices,:]
    p_RgXs_sorted = np.array(p_RgXs)[sort_indices]
    p_RgXs_maxinds = [np.argmax(np.abs(p_RgX_softmax[ib] - p_RgXs_sorted[ib])) for ib in range(N_b)]
    IB_dict_softmaxinit['p_RgX_maxdiff'] = np.array([p_RgX_softmax[ib].flatten()[p_RgXs_maxinds[ib]] - p_RgXs_sorted[ib].flatten()[p_RgXs_maxinds[ib]] for ib in range(N_b)])
    softmax_dict['IB_obj_fn'] = softmax_dict['I_XR'] - softmax_dict['beta'] * softmax_dict['I_RY']
    # IB_dict_randinit['IB_obj_fn'] = IB_dict_randinit['I_XR'] - IB_dict_randinit['beta'] * IB_dict_randinit['I_RY']
    IB_dict_softmaxinit['IB_obj_fn'] = IB_dict_softmaxinit['I_XR'] - IB_dict_softmaxinit['beta'] * IB_dict_softmaxinit['I_RY']

    return softmax_dict, IB_dict_randinit, IB_dict_softmaxinit